# ***ÁRBOLES DE DECISIÓN***

In [90]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

In [91]:
df = pd.read_csv("../data/diabetes_preprocesado.csv")

### 1- Separación Train-Test

In [92]:
# 1. Definimos las variables del modelo
# X: Matriz de características (features). Eliminamos la columna objetivo 'readmitted'.
X = df.drop('readmitted', axis=1)

# y: Vector objetivo (target). Contiene la clase que queremos predecir (0 o 1).
y = df['readmitted']

# 2. División del dataset en Entrenamiento (Train) y Prueba (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

### 2- Árbol sin restriccion

In [93]:
# 1. Configuración y entrenamiento del modelo sin restricciones.
# Se utiliza DecisionTreeClassifier con los parámetros por defecto.
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

# Generación de predicciones para calcular las métricas
y_pred = tree_full.predict(X_test)

# 2. Captura de métricas en formato de lista para el DataFrame.
# Se extraen las métricas de rendimiento y las características estructurales.
tree_baseline_data = [{
    'Configuración': 'Sin restricciones',
    'Train Accuracy': f"{tree_full.score(X_train, y_train):.4f}",
    'Test Accuracy': f"{accuracy_score(y_test, y_pred):.4f}",
    'Precision (Macro)': f"{precision_score(y_test, y_pred, average='macro'):.4f}",
    'Recall (Macro)': f"{recall_score(y_test, y_pred, average='macro'):.4f}",
    'F1 Score (Macro)': f"{f1_score(y_test, y_pred, average='macro'):.4f}",
    'Profundidad': tree_full.get_depth(),
    'N° Hojas': tree_full.get_n_leaves()
}]

# 3. Creación y visualización del DataFrame resumen.
df_tree_baseline = pd.DataFrame(tree_baseline_data)

# Visualización del DataFrame
df_tree_baseline

,Configuración,Train Accuracy,Test Accuracy,Precision (Macro),Recall (Macro),F1 Score (Macro),Profundidad,N° Hojas
0,Sin restricciones,1.0000,0.4875,0.3964,0.3972,0.3967,43,20921


Los resultados obtenidos con el árbol de decisión sin restricciones evidencian un claro problema de sobreajuste (*overfitting*). El modelo alcanza una precisión perfecta sobre el conjunto de entrenamiento (`Train Accuracy = 1.0000`), lo que significa que el árbol ha aprendido prácticamente de memoria todos los patrones y particularidades de los datos de entrenamiento. Sin embargo, esta capacidad de ajuste extremo no se traduce en un buen rendimiento sobre datos nuevos, ya que la precisión en el conjunto de prueba desciende hasta `0.4875`. Esta diferencia tan pronunciada entre entrenamiento y prueba indica que el modelo no consigue generalizar correctamente.

Al tratar un conjunto de datos relativamente grande y con numerosas variables categóricas transformadas durante el preprocesamiento, un árbol sin restricciones tiende a crecer excesivamente para capturar incluso pequeñas variaciones o ruido presente en los datos. Esto explica la elevada complejidad estructural observada en el modelo.

La profundidad alcanzada por el árbol (`43 niveles`) y el número de hojas (`20 921`) confirman este comportamiento. Un árbol tan profundo genera reglas de decisión extremadamente específicas, donde muchas hojas representan casos muy particulares del conjunto de entrenamiento. Aunque esto maximiza el rendimiento interno, provoca una pérdida importante de capacidad predictiva sobre nuevos pacientes. En problemas médicos como la predicción relacionada con diabetes, este comportamiento resulta especialmente problemático, ya que el objetivo principal es construir modelos robustos y generalizables, no simplemente memorizar ejemplos previos.

Las métricas de clasificación también reflejan un rendimiento limitado. La precisión macro (`0.3964`), el recall macro (`0.3972`) y el F1-score macro (`0.3967`) muestran valores relativamente bajos y equilibrados entre sí. Esto indica que el modelo tiene dificultades para clasificar adecuadamente las distintas clases del problema, posiblemente debido al desbalance de clases presente en la variable objetivo (`readmitted`). El hecho de utilizar métricas macro implica que todas las clases reciben la misma importancia, por lo que estos resultados sugieren que algunas categorías están siendo clasificadas de manera deficiente.

En conjunto, estos resultados demuestran que un árbol de decisión sin restricciones no es una solución adecuada para este problema, ya que produce un modelo excesivamente complejo y con escasa capacidad de generalización. Por ello, resulta necesario aplicar técnicas de regularización o poda, como limitar la profundidad máxima (`max_depth`), establecer un número mínimo de muestras por hoja (`min_samples_leaf`) o restringir el número mínimo de muestras para dividir nodos (`min_samples_split`). Estas estrategias permiten reducir la complejidad del árbol y mejorar el equilibrio entre aprendizaje y generalización.

### 3- Criterios de División

In [94]:
# 1. Definición de criterios y lista para resultados
criterios = ['gini', 'entropy', 'log_loss']
results_criterio = []

# 2. Bucle de entrenamiento y captura de métricas
for criterio in criterios:
    tree = DecisionTreeClassifier(criterion=criterio, random_state=42)
    
    # Validación cruzada (basada en Accuracy)
    cv = cross_val_score(tree, X_train, y_train, cv=5)
    
    # Entrenamiento final para este criterio
    tree.fit(X_train, y_train)
    
    # Predicciones para calcular métricas detalladas
    y_pred = tree.predict(X_test)
    
    # Guardamos en la lista
    results_criterio.append({
        'Criterio': criterio.capitalize(), 
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, y_pred):.4f}",
        'Precision (Macro)': f"{precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}",
        'Recall (Macro)': f"{recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}",
        'F1 Score (Macro)': f"{f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}",
        'N° Hojas': tree.get_n_leaves()
    })

# 3. Creación del DataFrame
df_criterios = pd.DataFrame(results_criterio)

# Mostramos la tabla final
df_criterios

,Criterio,CV Accuracy,Test Accuracy,Precision (Macro),Recall (Macro),F1 Score (Macro),N° Hojas
0,Gini,0.4830,0.4875,0.3964,0.3972,0.3967,20921
1,Entropy,0.4834,0.4838,0.3912,0.3918,0.3914,19852
2,Log_loss,0.4834,0.4838,0.3912,0.3918,0.3914,19852


El análisis de los distintos criterios de partición del árbol de decisión permite observar cómo varía el comportamiento del modelo en función de la medida utilizada para seleccionar las divisiones internas. En este caso, se evaluaron los criterios `gini`, `entropy` y `log_loss`, comparando su rendimiento mediante validación cruzada y métricas sobre el conjunto de prueba. Los resultados muestran que las diferencias entre los tres enfoques son relativamente pequeñas, lo que indica que el problema de clasificación asociado al dataset de diabetes no depende de manera significativa del criterio de impureza utilizado.

El criterio **Gini** presenta el mejor rendimiento general, aunque la diferencia respecto a los demás criterios es mínima. Su `CV Accuracy` de `0.4830` y `Test Accuracy` de `0.4875` reflejan un comportamiento ligeramente superior al obtenido con `entropy` y `log_loss`. Además, alcanza mejores valores en precisión, recall y F1-score macro (`≈0.396`), lo que sugiere una capacidad marginalmente más equilibrada para clasificar las distintas categorías de la variable objetivo. No obstante, el árbol generado continúa siendo extremadamente complejo, con `20 921 hojas`, lo que evidencia nuevamente un fuerte problema de sobreajuste.

Por otro lado, los criterios **Entropy** y **Log Loss** producen exactamente los mismos resultados tanto en validación cruzada como en las métricas de prueba. Esto ocurre porque ambos criterios están basados en conceptos similares derivados de la teoría de la información y, en muchos casos prácticos, generan árboles muy parecidos. Ambos modelos alcanzan un `CV Accuracy` de `0.4834` y un `Test Accuracy` de `0.4838`, valores ligeramente inferiores a los obtenidos con Gini. Asimismo, las métricas macro (`Precision`, `Recall` y `F1`) descienden ligeramente hasta aproximadamente `0.391`, indicando una capacidad predictiva algo menor.

Una diferencia relevante se encuentra en la complejidad estructural de los árboles generados. Mientras que Gini produce `20 921 hojas`, Entropy y Log Loss reducen este número a `19 852 hojas`. Aunque la reducción no es suficiente para solucionar el problema de sobreajuste, sí sugiere que estos criterios generan árboles ligeramente más compactos y menos fragmentados. Sin embargo, esta menor complejidad no se traduce en una mejora del rendimiento predictivo, por lo que el beneficio práctico resulta limitado.

Los resultados de validación cruzada son especialmente importantes en este análisis. Los tres criterios muestran valores muy similares (`≈0.483`), lo que indica que el comportamiento del modelo es consistente entre distintas particiones del conjunto de entrenamiento. Sin embargo, estas puntuaciones siguen siendo relativamente bajas, lo que refuerza la idea de que el árbol sin restricciones no logra capturar patrones verdaderamente generalizables dentro del dataset de diabetes.

En conjunto, el análisis permite concluir que el criterio de división tiene un impacto reducido sobre el rendimiento global del modelo. Aunque Gini ofrece resultados ligeramente mejores, ninguno de los criterios consigue resolver el principal problema del árbol: su excesiva complejidad y la falta de capacidad de generalización. Por ello, más allá de elegir un criterio de impureza concreto, resulta más importante aplicar técnicas de regularización y poda que permitan controlar el crecimiento del árbol y mejorar su rendimiento sobre datos no vistos.

### 4- Poda Previa: Grid Search de Hiperparámetros

In [95]:
# 1. Configuración de la malla de parámetros (Grid) y ejecución de la búsqueda.
param_grid = {
    'max_depth':         [3, 5, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 5, 10],
    'max_leaf_nodes':    [None, 10, 20, 50]
}

grid_tree = GridSearchCV(
    DecisionTreeClassifier(criterion='gini', random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    return_train_score=False
)

grid_tree.fit(X_train, y_train)

# 2. Generación de tabla comparativa
results_data = []
cv_res = grid_tree.cv_results_
paso = len(cv_res['params']) // 10 

for i in range(0, len(cv_res['params']), paso):
    params = cv_res['params'][i]
    
    model_tmp = DecisionTreeClassifier(**params, random_state=42)
    model_tmp.fit(X_train, y_train)
    
    # Predicciones para métricas detalladas
    y_pred = model_tmp.predict(X_test)
    
    results_data.append({
        'max_depth': params['max_depth'],
        'min_samples_split': params['min_samples_split'],
        'min_samples_leaf': params['min_samples_leaf'],
        'max_leaf_nodes': params['max_leaf_nodes'],
        'CV Acc': f"{cv_res['mean_test_score'][i]:.4f}",
        'Test Acc': f"{accuracy_score(y_test, y_pred):.4f}",
        'Precision': f"{precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}",
        'Recall': f"{recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}",
        'F1 Score': f"{f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}"
    })

df_pre_poda = pd.DataFrame(results_data)

# 3. Identificación y registro del mejor modelo
mejor_params = grid_tree.best_params_
mejor_modelo = grid_tree.best_estimator_
y_pred_mejor = mejor_modelo.predict(X_test)

mejor_fila = pd.DataFrame([{
    'max_depth': mejor_params['max_depth'],
    'min_samples_split': mejor_params['min_samples_split'],
    'min_samples_leaf': mejor_params['min_samples_leaf'],
    'max_leaf_nodes': mejor_params['max_leaf_nodes'],
    'CV Acc': f"{grid_tree.best_score_:.4f}",
    'Test Acc': f"{accuracy_score(y_test, y_pred_mejor):.4f}",
    'Precision': f"{precision_score(y_test, y_pred_mejor, average='macro', zero_division=0):.4f}",
    'Recall': f"{recall_score(y_test, y_pred_mejor, average='macro', zero_division=0):.4f}",
    'F1 Score': f"{f1_score(y_test, y_pred_mejor, average='macro', zero_division=0):.4f}"
}], index=['Mejor'])

# 4. Consolidación del reporte final
df_reporte = pd.concat([df_pre_poda, mejor_fila])
df_reporte

C:\Users\elena\AppData\Local\Temp\ipykernel_16732\49684475.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_reporte = pd.concat([df_pre_poda, mejor_fila])


,max_depth,min_samples_split,min_samples_leaf,max_leaf_nodes,CV Acc,Test Acc,Precision,Recall,F1 Score
0,3.0,2,1,NaN,0.3749,0.5586,0.3535,0.3938,0.3692
1,3.0,5,5,10.0,0.3749,0.5586,0.3535,0.3938,0.3692
2,3.0,10,1,50.0,0.3749,0.5586,0.3535,0.3938,0.3692
3,5.0,20,5,NaN,0.3806,0.5751,0.4821,0.3990,0.3723
4,5.0,2,2,20.0,0.3756,0.5750,0.3646,0.3971,0.3687
5,5.0,5,10,50.0,0.3803,0.5751,0.4821,0.3990,0.3723
6,10.0,10,2,10.0,0.3659,0.5586,0.3535,0.3938,0.3692
7,10.0,20,10,20.0,0.3803,0.5750,0.3646,0.3971,0.3687
8,NaN,2,5,NaN,0.4002,0.5161,0.4120,0.4035,0.4046
9,NaN,5,1,20.0,0.3803,0.5750,0.3646,0.3971,0.3687


Los resultados obtenidos mediante la aplicación de poda previa muestran una mejora significativa respecto al árbol sin restricciones analizado anteriormente. En este caso, se utilizaron diferentes combinaciones de hiperparámetros para controlar el crecimiento del árbol antes de su construcción, limitando aspectos como la profundidad máxima (`max_depth`), el número mínimo de muestras necesarias para dividir un nodo (`min_samples_split`), el número mínimo de muestras por hoja (`min_samples_leaf`) y el número máximo de hojas (`max_leaf_nodes`). El objetivo de esta estrategia es reducir la complejidad del modelo y evitar el sobreajuste observado en el árbol sin restricciones.

En términos generales, los resultados evidencian que restringir el crecimiento del árbol mejora considerablemente la capacidad de generalización. Mientras que el árbol sin restricciones obtenía un `Test Accuracy` cercano al `48%`, los modelos con poda previa alcanzan valores entre `55%` y `58%`, lo que representa una mejora importante sobre datos no vistos. Este comportamiento indica que la reducción de complejidad permite al modelo capturar patrones más generales y menos dependientes del ruido presente en el conjunto de entrenamiento.

Los modelos más simples, especialmente aquellos con `max_depth = 3`, presentan resultados muy estables. Independientemente de los valores de `min_samples_split`, `min_samples_leaf` o `max_leaf_nodes`, las métricas obtenidas son prácticamente idénticas. Esto sugiere que una profundidad tan reducida limita fuertemente la capacidad del árbol, haciendo que otros hiperparámetros tengan un impacto mínimo. Estos modelos alcanzan un `Test Accuracy` de aproximadamente `0.5586`, pero presentan métricas macro relativamente bajas (`F1 ≈ 0.3692`), indicando que aunque clasifican correctamente una parte importante de los datos, continúan teniendo dificultades para equilibrar el rendimiento entre todas las clases.

A medida que aumenta la profundidad máxima a `5` o `10`, el modelo gana capacidad predictiva y mejora ligeramente las métricas de clasificación. Algunos modelos alcanzan valores de precisión superiores (`Precision ≈ 0.48`), manteniendo un `Test Accuracy` cercano al `57.5%`. Esto indica que árboles moderadamente más profundos consiguen representar mejor la complejidad de los datos clínicos asociados a la diabetes sin caer inmediatamente en un sobreajuste extremo.

Uno de los aspectos más relevantes del análisis es el comportamiento de los modelos sin límite explícito de profundidad (`max_depth = None`). Cuando no se controla adecuadamente el crecimiento del árbol, pero sí se restringe mediante `min_samples_leaf` o `max_leaf_nodes`, el rendimiento mejora de forma notable. En particular, la configuración con `min_samples_split = 10`, `min_samples_leaf = 5` y `max_leaf_nodes = 50` obtiene un `Test Accuracy` de `0.5844` y una precisión macro de `0.5591`, siendo uno de los mejores resultados de la tabla. Esto demuestra que limitar el número de muestras por nodo y restringir la fragmentación excesiva del árbol puede ser más efectivo que simplemente imponer una profundidad máxima fija.

El mejor modelo identificado mediante `GridSearchCV` corresponde a la configuración:

* `max_depth = 10`
* `min_samples_split = 2`
* `min_samples_leaf = 10`
* `max_leaf_nodes = None`

Este modelo alcanza un `F1 Score` macro de `0.4199`, superior al resto de configuraciones evaluadas, junto con un `Test Accuracy` de `0.5812`. Además, obtiene el mejor resultado de validación cruzada (`CV Acc = 0.4153`), lo que indica una mayor estabilidad y capacidad de generalización respecto a los demás modelos. El hecho de utilizar un valor elevado de `min_samples_leaf = 10` obliga al árbol a crear hojas más representativas y menos específicas, reduciendo significativamente el sobreajuste.

Las métricas macro continúan siendo relativamente moderadas, especialmente el recall (`0.4284`) y el F1-score (`0.4199`). Esto sugiere que el problema de clasificación sigue siendo complejo, probablemente debido al desbalance de clases presente en el dataset de diabetes y a la dificultad inherente de distinguir correctamente todos los tipos de readmisión hospitalaria. Sin embargo, comparado con el árbol sin restricciones, el modelo podado muestra una mejora clara tanto en estabilidad como en capacidad predictiva.

En conjunto, los resultados demuestran que la poda previa constituye una estrategia efectiva para controlar la complejidad de los árboles de decisión. La limitación del crecimiento del árbol permite reducir el sobreajuste, mejorar la generalización y obtener modelos más equilibrados y robustos. Además, el análisis evidencia que hiperparámetros como `min_samples_leaf` y `max_depth` tienen un impacto especialmente importante en el rendimiento final del modelo, siendo fundamentales para lograr un equilibrio adecuado entre simplicidad y capacidad predictiva.

### 5-  Poda Posterior: Cost-Complexity Pruning

In [96]:
# 1. Obtención del camino de poda.
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]  

results_ccp = []

# 2. Selección de valores representativos de alpha
indices = np.unique(np.linspace(0, len(ccp_alphas) - 1, 8).astype(int))
alphas_to_test = ccp_alphas[indices]

# 3. Bucle de evaluación para cada alpha
for alpha in alphas_to_test:
    tree = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    
    # Validación Cruzada
    cv = cross_val_score(tree, X_train, y_train, cv=5)
    
    # Ajuste del modelo
    tree.fit(X_train, y_train)
    
    # Generación de predicciones
    y_pred = tree.predict(X_test)
    
    # Registro de métricas detalladas
    results_ccp.append({
        'ccp_alpha': f"{alpha:.5f}",
        'CV Accuracy': f"{cv.mean():.4f}",
        'Test Accuracy': f"{accuracy_score(y_test, y_pred):.4f}",
        'Precision': f"{precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}",
        'Recall': f"{recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}",
        'F1 Score': f"{f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}",
        'Profundidad': tree.get_depth(),
        'N° Hojas': tree.get_n_leaves()
    })

# 4. Creación del reporte y etiquetado
df_ccp = pd.DataFrame(results_ccp)
idx_mejor = df_ccp['CV Accuracy'].astype(float).idxmax()

nuevos_indices = {i: str(i) for i in range(len(df_ccp))}
nuevos_indices[0] = "0.0 (sin poda)"
nuevos_indices[idx_mejor] = "Mejor"
nuevos_indices[len(df_ccp)-1] = "alpha grande"

df_ccp.index = nuevos_indices.values()

# Visualización
df_ccp

,ccp_alpha,CV Accuracy,Test Accuracy,Precision,Recall,F1 Score,Profundidad,N° Hojas
0.0 (sin poda),0.00000,0.4830,0.4875,0.3964,0.3972,0.3967,43,20921
1,0.00001,0.4832,0.4958,0.3995,0.3987,0.3990,41,17611
2,0.00002,0.4925,0.4984,0.4017,0.4005,0.4009,39,15522
3,0.00002,0.4944,0.5004,0.4007,0.3992,0.3997,39,13161
4,0.00002,0.4955,0.5114,0.4072,0.4034,0.4041,39,10376
5,0.00003,0.5036,0.5245,0.4179,0.4093,0.4103,37,7617
6,0.00003,0.5238,0.5412,0.4366,0.4146,0.4149,30,3975
alpha grande,0.00428,0.5557,0.5495,0.3486,0.3910,0.3673,1,2


Los resultados obtenidos mediante la poda posterior muestran de forma clara cómo el parámetro de complejidad `ccp_alpha` influye directamente en el equilibrio entre complejidad y capacidad de generalización del árbol de decisión. A diferencia de la poda previa, donde las restricciones se aplican antes de construir el árbol, la poda posterior parte de un árbol completamente desarrollado y posteriormente elimina ramas poco relevantes según su contribución al rendimiento del modelo. Esta técnica busca simplificar la estructura del árbol sin perder información predictiva importante.

El primer resultado, correspondiente a `ccp_alpha = 0.00000`, representa el árbol sin poda previamente analizado. Este modelo alcanza un `Test Accuracy` de `0.4875`, una profundidad de `43` niveles y un total de `20 921 hojas`. Estas cifras reflejan nuevamente un modelo extremadamente complejo y con un fuerte problema de sobreajuste, ya que el árbol memoriza excesivamente los datos de entrenamiento sin generalizar adecuadamente sobre nuevos ejemplos.

A medida que aumenta ligeramente el valor de `ccp_alpha`, el árbol comienza a reducir progresivamente su complejidad. Con valores pequeños como `0.00001` y `0.00002`, se observa una disminución gradual tanto en la profundidad como en el número de hojas. Por ejemplo, el árbol pasa de más de `20 000 hojas` a aproximadamente `15 522` y posteriormente `10 376 hojas`. Esta simplificación estructural viene acompañada de una mejora constante en las métricas de rendimiento. El `Test Accuracy` aumenta desde `0.4875` hasta `0.5114`, mientras que el `F1 Score` macro mejora ligeramente hasta `0.4041`. Esto indica que eliminar ramas poco relevantes ayuda al modelo a reducir el ruido y mejorar su capacidad de generalización.

Uno de los resultados más interesantes aparece con valores intermedios de poda, especialmente alrededor de `ccp_alpha = 0.00003`. En estos casos, el modelo alcanza sus mejores métricas de equilibrio entre complejidad y rendimiento. El árbol identificado como mejor candidato obtiene un `CV Accuracy` de `0.5238`, un `Test Accuracy` de `0.5412` y un `F1 Score` macro de `0.4149`. Además, la profundidad del árbol se reduce considerablemente hasta `30 niveles` y el número de hojas desciende a `3 975`, lo que supone una reducción muy significativa respecto al árbol original.

Este comportamiento evidencia que la poda posterior consigue eliminar una gran cantidad de ramas redundantes o demasiado específicas, manteniendo únicamente aquellas divisiones que realmente aportan capacidad predictiva. La mejora en validación cruzada demuestra además que el modelo podado es más estable y generaliza mejor sobre distintas particiones de los datos. En el contexto del dataset de diabetes, esto resulta especialmente importante, ya que permite construir reglas de decisión más robustas y menos sensibles a patrones particulares del conjunto de entrenamiento.

Sin embargo, cuando el valor de `ccp_alpha` se incrementa excesivamente (`0.00428`), el árbol se simplifica en exceso. En este caso, la profundidad se reduce drásticamente a solo `1 nivel` y el modelo queda compuesto por únicamente `2 hojas`. Aunque el `CV Accuracy` alcanza el valor más alto (`0.5557`) y el `Test Accuracy` sigue siendo relativamente elevado (`0.5495`), las métricas macro empeoran notablemente (`F1 Score = 0.3673`). Esto indica que el modelo ha perdido demasiada capacidad de representación y probablemente está clasificando mayoritariamente la clase dominante, ignorando patrones importantes de las clases minoritarias.

Este fenómeno refleja el problema opuesto al sobreajuste: el *underfitting* o subajuste. Un árbol excesivamente simple no dispone de suficiente capacidad para capturar la complejidad inherente de los datos clínicos asociados a la diabetes. Por ello, aunque aparentemente mantiene una precisión aceptable, su capacidad real para discriminar correctamente entre clases se reduce significativamente.

En conjunto, los resultados muestran que la poda posterior es una estrategia efectiva para mejorar el rendimiento de los árboles de decisión. El incremento gradual de `ccp_alpha` permite controlar la complejidad del modelo y reducir el sobreajuste observado en el árbol sin restricciones.

### 6- Importancia de características

In [ ]:
# 1. Obtención de las importancias desde el mejor modelo encontrado.
# Se utiliza el estimador óptimo obtenido en el GridSearchCV (grid_tree).
# feature_importances_ asigna un valor a cada variable: cuanto más alto, 
# más contribuye esa variable a la pureza de los nodos.
best_tree = grid_tree.best_estimator_
importances = best_tree.feature_importances_
feature_names = X.columns.tolist()

# 2. Ordenación de las variables.
indices = importances.argsort()[::-1]

# 3. Construcción de la estructura de datos para el ranking.
ranking_data = []
for i, idx in enumerate(indices):
    ranking_data.append({
        'Ranking': i + 1,
        'Feature': feature_names[idx],
        'Importancia (Gini)': f"{importances[idx]:.4f}"
    })


# 4. Creación y visualización del dataframe de importancia
df_importancia = pd.DataFrame(ranking_data)

# Visualización del resultado en formato tabla.
df_importancia

,Ranking,Feature,Importancia (Gini)
0,1,number_inpatient,0.2582
1,2,encounter_id,0.1769
2,3,discharge_disposition_id,0.1369
3,4,patient_nbr,0.1300
4,5,diag_1,0.0340
5,6,number_diagnoses,0.0277
6,7,age,0.0262
7,8,num_medications,0.0256
8,9,number_emergency,0.0250
9,10,number_outpatient,0.0218


El análisis de importancia de variables permite identificar qué características del dataset de diabetes tienen mayor influencia en las decisiones tomadas por el árbol de decisión optimizado. En este caso, las importancias se calcularon utilizando la reducción de impureza Gini (*Gini Importance*), donde un valor más alto indica que la variable contribuye con mayor frecuencia y efectividad a dividir los datos en nodos más puros. Este análisis resulta especialmente útil para interpretar el modelo y comprender qué factores son más relevantes en la predicción de la readmisión hospitalaria de pacientes diabéticos.

La variable más importante del modelo es `number_inpatient`, con una importancia de `0.2582`, muy superior al resto de características. Esta variable representa el número de ingresos hospitalarios previos del paciente y su elevada relevancia tiene sentido desde un punto de vista clínico, ya que un historial frecuente de hospitalizaciones suele estar asociado a pacientes con mayor gravedad, complicaciones médicas o peor control de la diabetes. El árbol utiliza esta información como uno de los principales criterios para estimar la probabilidad de readmisión.

En segundo lugar, aparecen `encounter_id` (`0.1769`) y `patient_nbr` (`0.1300`). Aunque estas variables presentan una importancia alta, su interpretación debe realizarse con cuidado. Ambas son identificadores administrativos y, en teoría, no deberían contener información predictiva directa sobre el estado clínico del paciente. Su elevada relevancia puede indicar que el modelo está capturando patrones específicos asociados a determinados registros o pacientes concretos, lo que podría ser una señal de sobreajuste residual o de fuga de información (*data leakage*). En un escenario real de modelado, estas variables normalmente deberían eliminarse antes del entrenamiento para evitar que el modelo aprenda relaciones artificiales no generalizables.

La variable `discharge_disposition_id` ocupa la tercera posición (`0.1369`) y representa el tipo de alta hospitalaria del paciente. Esta característica tiene una interpretación clínica mucho más coherente, ya que el destino del paciente tras el alta (domicilio, traslado, cuidados especializados, etc.) puede reflejar el nivel de gravedad del caso y su riesgo de reingreso. Del mismo modo, variables diagnósticas como `diag_1`, `diag_2` y `diag_3` también muestran cierta relevancia, indicando que los diagnósticos médicos asociados al paciente influyen en la probabilidad de readmisión.

Entre las variables con importancia intermedia destacan `number_diagnoses`, `age`, `num_medications`, `number_emergency`, `number_outpatient` y `num_lab_procedures`. Estas características están relacionadas con la complejidad clínica y el uso del sistema sanitario por parte del paciente. Por ejemplo, un mayor número de diagnósticos o medicamentos puede reflejar pacientes con múltiples enfermedades crónicas, mientras que las visitas frecuentes a urgencias o consultas ambulatorias pueden indicar un estado de salud más inestable. La edad también aparece como un factor relevante, algo esperable en enfermedades crónicas como la diabetes, donde el riesgo de complicaciones aumenta con el envejecimiento.

Otras variables relacionadas con tratamientos específicos para la diabetes, como `insulin`, `metformin`, `glyburide` o `glipizide`, presentan importancias relativamente bajas. Esto sugiere que, aunque la medicación tiene cierta influencia, el modelo considera más determinantes los patrones generales de hospitalización y estado clínico que los tratamientos farmacológicos individuales. Asimismo, variables demográficas como `race` y `gender` tienen una contribución muy reducida, indicando que el árbol apenas utiliza estas características para tomar decisiones relevantes.

Un aspecto especialmente llamativo es que numerosas variables poseen una importancia igual a `0.0000`. Esto significa que el árbol nunca las utilizó para realizar divisiones durante el entrenamiento. Entre ellas se encuentran varios medicamentos poco frecuentes (`troglitazone`, `examide`, `citoglipton`, entre otros). Esto puede deberse a dos motivos principales: o bien dichas variables tienen muy poca variabilidad dentro del dataset, o bien no aportan información útil para mejorar la clasificación del modelo.

En conjunto, el ranking de importancia muestra que el modelo basa principalmente sus decisiones en variables relacionadas con el historial hospitalario del paciente, la complejidad clínica y la gravedad de los casos. Sin embargo, también revela posibles problemas metodológicos asociados al uso de identificadores administrativos (`encounter_id` y `patient_nbr`), los cuales podrían introducir sesgos o fuga de información.